# Validation

In [26]:
SOLUTIONS_FILE = './data/validation/solutions.json'
EVALUATIONS_FILE = './data/validation/evaluations.json'
MAX_WORKERS = 5
N_JUDGES = 5
GEN_MODEL = 'deepseek/deepseek-v4-flash'
EVAL_MODEL = 'openai/gpt-4.1'
TEMPERATURE = 0.5

PROBLEM = """
Создай мастер-план города Гатчина
"""

In [27]:
from src import read_json, write_json, generate_id

solutions = read_json(SOLUTIONS_FILE) or {}
evaluations = read_json(EVALUATIONS_FILE) or {}

In [28]:
from langchain_openai import ChatOpenAI
from fp2mp_baselines.config import config

llm = ChatOpenAI(
    model=GEN_MODEL,
    base_url=config.base_url,
    api_key=config.api_key,
    temperature=TEMPERATURE,
)

In [29]:
from fp2mp_core.graph import run as fp2mp_run
from fp2mp_core.ablation import AblationConfig

class FP2MPWrapper():

    def __init__(self, model : str, max_iterations : int, **ablation_kwargs):
        self.model = model
        self.max_iterations = max_iterations
        self.cfg = AblationConfig(**ablation_kwargs)

    def invoke(self, state):
        input = state['input']
        model = self.model
        max_iterations = self.max_iterations
        cfg = self.cfg
        return fp2mp_run(input, model, max_iterations, ablation=cfg)

In [30]:
from fp2mp_core.tools.code_exec import execute_python_tool, list_available_libraries_tool, check_available_data_tool 
from fp2mp_core.tools.web_search import fetch_url_tool, web_search_tool

tools = [
    execute_python_tool, list_available_libraries_tool, check_available_data_tool,
    fetch_url_tool, web_search_tool
]

from fp2mp_baselines.single_agent import build_single_agent_graph
from fp2mp_baselines.cot import build_cot_graph
from fp2mp_baselines.react import build_react_graph
from fp2mp_baselines.generator_critic import build_generator_critic_graph
from fp2mp_baselines.debate import build_debate_graph
from fp2mp_baselines.major_vote import build_major_vote_graph

baselines = {
    'single-agent' : build_single_agent_graph(llm),
    'chain-of-thoughts': build_cot_graph(llm),
    'react': build_react_graph(llm, tools=tools), 
    'generator-critic': build_generator_critic_graph(llm),
    'debate': build_debate_graph(llm, num_agents=3, debate_rounds=3),
    'major-vote': build_major_vote_graph(llm, num_agents=3),
    'fp2mp': FP2MPWrapper(GEN_MODEL, max_iterations=3),
}

## Solutions

In [31]:
tasks = []

for baseline_name in baselines.keys():
    solution_id = generate_id(PROBLEM, baseline_name, GEN_MODEL)
    if solution_id not in solutions:
        tasks.append({
            'solution_id': solution_id,
            'problem': PROBLEM,
            'baseline_name': baseline_name
        })

len(tasks)

0

In [32]:
from tqdm.contrib.concurrent import thread_map

def solve_task(task : dict[str,str]):
    solution_id = task['solution_id']
    problem = task['problem']
    baseline_name = task['baseline_name']

    baseline_graph = baselines[baseline_name]

    state = {'input': problem}

    graph_state = None
    while graph_state is None:
        try:
            graph_state = baseline_graph.invoke(state)
            return {
                'solution_id': solution_id,
                'solution_data': {
                    'problem': problem,
                    'baseline': baseline_name,
                    'model': GEN_MODEL,
                    'content': graph_state['output'],
                    'log': [m.model_dump() for m in graph_state['log']]
                }
            }   
        except:
            ... 

results = thread_map(solve_task, tasks, max_workers=MAX_WORKERS)

0it [00:00, ?it/s]

In [33]:
for result in results:
    solution_id = result['solution_id']
    solution_data = result['solution_data']
    solutions[solution_id] = solution_data

write_json(solutions, SOLUTIONS_FILE)

## Evaluation

In [34]:
tasks = []

for solution_id, solution_data in solutions.items():
    evaluation_id = generate_id(solution_id, EVAL_MODEL)
    if evaluation_id not in evaluations:
        problem = solution_data['problem']
        solution_content = solution_data['content']
        tasks.append({
            'evaluation_id': evaluation_id,
            'solution_id': solution_id,
            'problem': problem,
            'solution_content': solution_content
        })

len(tasks)

7

In [35]:
from tqdm import tqdm
from fp2mp_eval.core import FP2MPEval

fp2mp_eval = FP2MPEval(EVAL_MODEL, n_judges=N_JUDGES)

for task in tqdm(tasks):
    evaluation_id = task['evaluation_id']
    solution_id = task['solution_id']
    problem = task['problem']
    solution_content = task['solution_content']

    case = (problem, solution_content)
    evals = fp2mp_eval.evaluate_case(case, MAX_WORKERS)
    
    evaluations[evaluation_id] = {
        'solution_id': solution_id,
        'model': EVAL_MODEL,
        'content': [e.model_dump() for e in evals]
    }

100%|██████████| 7/7 [01:26<00:00, 12.33s/it]


In [36]:
write_json(evaluations, EVALUATIONS_FILE)

## Experts

todo